# Merge & Decode Facebook Messages
Reads all `message_*.json` files from `Data/PrivateData/`, merges them, sorts by `timestamp_ms`, fixes the Facebook Latin-1-escaped UTF-8 encoding, and writes the result to `Data/EncodedData/messages.json`.

In [1]:
import json
import glob
import os
from pathlib import Path

In [2]:
def fix_encoding(obj):
    """Recursively fix Facebook's mojibake: latin-1 bytes re-encoded as UTF-8."""
    if isinstance(obj, str):
        try:
            return obj.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return obj
    if isinstance(obj, list):
        return [fix_encoding(item) for item in obj]
    if isinstance(obj, dict):
        return {fix_encoding(k): fix_encoding(v) for k, v in obj.items()}
    return obj

In [3]:
base_dir = Path(os.getcwd()).parent  # messagesAnalysis/
input_dir = base_dir / 'Data' / 'PrivateData'
output_dir = base_dir / 'Data' / 'EncodedData'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input : {input_dir}")
print(f"Output: {output_dir}")

Input : /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/PrivateData
Output: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData


In [4]:
input_files = sorted(input_dir.glob('message_*.json'))
print(f"Found {len(input_files)} file(s): {[f.name for f in input_files]}")

all_messages = []
participants_set = {}

for path in input_files:
    # Read raw bytes and parse — Facebook JSON is valid JSON but strings are mojibake
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = fix_encoding(data)

    # Collect unique participants keyed by name
    for p in data.get('participants', []):
        participants_set[p['name']] = p

    all_messages.extend(data.get('messages', []))

print(f"Total messages before dedup: {len(all_messages)}")

Found 5 file(s): ['message_1.json', 'message_2.json', 'message_3.json', 'message_4.json', 'message_5.json']
Total messages before dedup: 44422


In [5]:
# Sort ascending by timestamp_ms (oldest first)
all_messages.sort(key=lambda m: m.get('timestamp_ms', 0))

merged = {
    'participants': list(participants_set.values()),
    'messages': all_messages,
}

print(f"Participants : {len(merged['participants'])}")
print(f"Total messages: {len(merged['messages'])}")
print(f"Oldest : {merged['messages'][0].get('timestamp_ms')}")
print(f"Newest : {merged['messages'][-1].get('timestamp_ms')}")

Participants : 37
Total messages: 44422
Oldest : 1768955883993
Newest : 1780454758698


In [6]:
output_path = output_dir / 'messages.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"Written to {output_path}  ({output_path.stat().st_size / 1024:.1f} KB)")

Written to /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages.json  (16140.7 KB)


In [7]:
# Sanity-check: print first 3 messages
for msg in merged['messages'][:3]:
    print(msg)

{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955883993, 'content': 'Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!', 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}
{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955890861, 'content': 'Yoroshiku cả nhà ạ', 'reactions': [{'reaction': '❤', 'actor': 'Linh Tran Hoang'}], 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}
{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955908508, 'content': 'Nhật đã đặt tên nhóm là MPKEN | "Lịch sử & Triết học".', 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}
